# Persistence (Baseline) Forecast
This notebook implements a simple persistence (naive) forecast for the National ILINet % weighted ILI series, splits the data into a training and holdout set, computes RMSE on the holdout, and plots results.

In [13]:
# Load libraries
library(tidyverse)
library(readr)
library(lubridate)

options(repr.plot.width = 12, repr.plot.height = 6, repr.plot.res = 150)

# Parameters
holdout_weeks <- 104  # adjust as desired (e.g. 52 or 104)
season_lag_weeks <- 104 # approx. 2 years (52*2). Adjust if you prefer a date-based join

# Read cleaned ILINet (created by data_cleaning.ipynb)
ilinet <- readr::read_rds('fluview_clean/ilinet_clean.rds')

# Prepare national weekly series
df <- ilinet %>%
  filter(region_type == 'National') %>%
  arrange(week_start) %>%
  select(week_start, percent_weighted_ili) %>%
  mutate(week_start = as.Date(week_start))

# Drop rows with missing target values (we need observed history for persistence/seasonal forecasts)
df <- df %>% filter(!is.na(percent_weighted_ili))

n <- nrow(df)
if (n <= 1) stop('Not enough data after removing NA values')
if (holdout_weeks >= n) stop('holdout_weeks must be smaller than the number of observations')

# Create simple lag for persistence (previous week) and an indexed lag for seasonal-naive (2 years)\n
df <- df %>% mutate(y = percent_weighted_ili, y_lag = lag(y, 1))

# For seasonal-naive (same week 2 years ago) we'll use a 104-week lag as an approximation\n
df <- df %>% mutate(y_lag_2yr = lag(y, season_lag_weeks))

# Alternate approach (commented): align by date exactly using lubridate::years(2) then join.\n
# df2 <- df %>% select(week_start, y) %>% rename(week_ref = week_start, y_ref = y)\n
# df <- df %>% mutate(week_ref = week_start - years(2)) %>% left_join(df2, by = c('week_ref' = 'week_start'))\n

# Split into training and holdout by last `holdout_weeks` observations
train_idx <- 1:(n - holdout_weeks)
test_idx  <- (n - holdout_weeks + 1):n
train <- df[train_idx, ]
holdout <- df[test_idx, ]

# Persistence forecast for holdout: forecast_t = observed_{t-1} (y_lag)
holdout <- holdout %>% mutate(persistence_forecast = y_lag)

# Seasonal-naive forecast for holdout: value from same week ~2 years ago (y_lag_2yr)
holdout <- holdout %>% mutate(seasonal2y_forecast = y_lag_2yr)

# Compute RMSE on the holdout (exclude NA forecasts)
rmse <- function(obs, pred) {
  sqrt(mean((obs - pred)^2, na.rm = TRUE))
}
holdout_rmse_persistence <- rmse(holdout$y, holdout$persistence_forecast)
holdout_rmse_seasonal2y  <- rmse(holdout$y, holdout$seasonal2y_forecast)

print(paste0('Holdout RMSE (persistence): ', round(holdout_rmse_persistence, 4)))
print(paste0('Holdout RMSE (seasonal-naive ~2yr): ', round(holdout_rmse_seasonal2y, 4)))

[1] "Holdout RMSE (persistence): 0.5334"
[1] "Holdout RMSE (seasonal-naive ~2yr): 1.6186"


In [ ]:
# Quick plot: observed vs persistence and seasonal forecasts on the holdout period
library(ggplot2)
plot_df <- bind_rows(
  train %>% select(week_start, y) %>% mutate(type = 'train'),
  holdout %>% select(week_start, y) %>% mutate(type = 'holdout')
)

ggplot(plot_df, aes(x = week_start)) +
  geom_line(aes(y = y, color = type), size = 0.8) +
  geom_point(data = holdout, aes(x = week_start, y = persistence_forecast), color = 'red', size = 1.5, alpha = 0.8) +
  geom_point(data = holdout, aes(x = week_start, y = seasonal2y_forecast), color = 'blue', size = 1.5, alpha = 0.8) +
  labs(title = sprintf('Forecasts vs observed (RMSE persistence=%.3f, seasonal2y=%.3f)', holdout_rmse_persistence, holdout_rmse_seasonal2y),
       x = 'Week start', y = '% Weighted ILI') +
  scale_color_manual(values = c('train' = 'black', 'holdout' = 'darkgrey')) +
  theme_minimal()

ERROR: Error in parse(text = input): <text>:18:1: unexpected ':'
17: 
18: :
    ^
